# Simple neural network with Keras

In this lab, we will build with Keras neural networks that recognize MNIST's handwritten digits:

1. a multinomial logistic regression, the network without a hidden layer;
2. a regularized multilayer perceptron;
3. a hyperparameter search with KerasTuner;
4. monitoring the training with TensorBoard, early stopping and saving the best model.

You don't need a GPU for this lab: everything is fast on CPU. You can therefore switch runtime if the one you were given has a GPU (`Runtime > Change runtime type`).

In [ ]:
!pip install -q keras-tuner
import keras
import keras_tuner
import matplotlib.pyplot as plt
import numpy as np
from keras import layers

# Same random draws on every run: reproducible results
keras.utils.set_random_seed(0)

## Data

This time, we load the dataset through Keras.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

## Have a look at the data

Study how the data is stored:

- the shape and type of the four arrays;
- the value of the first labels and the range of the pixels;
- a few images, with their label;
- the number of examples of each class: are the classes balanced?

**Useful documentation**

- [`numpy.bincount`](https://numpy.org/doc/stable/reference/generated/numpy.bincount.html)
- [`matplotlib.axes.Axes.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html)

In [ ]:
# Your code here

### Additional hints

The attributes and calls to use:

- `X_train.shape`, `X_train.dtype`, `X_train.min()` and `X_train.max()`
- `np.bincount(y_train)`: the number of occurrences of each integer, from 0 to 9
- `plt.subplots(1, 10)` returns a figure and 10 axes; `ax.imshow(image, cmap="gray_r")` shows a 28 × 28 grayscale image

A skeleton to copy into the exercise cell, replacing each `...`:

```python
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(X_train.dtype, X_train.min(), X_train.max(), y_train[:10])
print(np.bincount(...))

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(..., cmap="gray_r")
  ax.set_title(...)
  ax.axis("off")
plt.show()
```

### Solution

In [ ]:
print(f"Shape of X_train: {X_train.shape}, of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}, of y_test: {y_test.shape}")
print(f"Image type: {X_train.dtype}, from {X_train.min()} to {X_train.max()}")
print(f"Label type: {y_train.dtype}, first ones: {y_train[:10]}")
print(f"Examples per class: {np.bincount(y_train)}")

# 60,000 training images and 10,000 test images, of 28 × 28 pixels; 8-bit unsigned
# integers (uint8), from 0 to 255; integer labels, from 0 to 9; from 5,421 to 6,742
# examples per class: roughly balanced classes

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(X_train[i], cmap="gray_r")
  ax.set_title(y_train[i])
  ax.axis("off")
plt.show()

## Data transformation

We need to transform this data:

1. Flatten the images: an array of shape `(_, 784)` rather than `(_, 28, 28)`, where `_` is the number of examples. Each image becomes a vector of 784 pixels, the input a `Dense` layer expects.
2. Bring the pixels into the $[0, 1]$ range rather than $[0, 255]$.
3. [One-hot](https://en.wikipedia.org/wiki/One-hot) encode the classes: the first of the two losses we will see, `categorical_crossentropy`, expects 10 values per example, all zero except for a 1 at the index of its class.

Apply the first two transformations to `X_train` and `X_test`. For the third one, keep `y_train` and `y_test`, and store the encoded classes in `Y_train` and `Y_test`.

**Useful documentation**

- [`keras.utils.to_categorical`](https://keras.io/api/utils/python_utils/#tocategorical-function)

In [ ]:
nb_classes = 10
input_dim = 28 * 28

# Your code here

### Additional hints

The calls and parameters to use:

- `X_train.reshape(-1, input_dim)`: `-1` lets NumPy compute the number of examples
- `/ 255` divides each pixel and gives floats
- `keras.utils.to_categorical(y_train, num_classes=nb_classes)` returns an array of shape `(60000, 10)`; label 5 becomes `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]` in it

A skeleton to copy into the exercise cell, replacing each `...`:

```python
X_train = X_train.reshape(...) / ...
X_test = ...
Y_train = keras.utils.to_categorical(..., num_classes=nb_classes)
Y_test = ...
```

### Solution

In [ ]:
# "Flat" images: the network's input is a vector of input_dim pixels, in [0, 1].
# -1 lets NumPy compute the number of examples.
X_train = X_train.reshape(-1, input_dim) / 255
X_test = X_test.reshape(-1, input_dim) / 255

# The classes as one-hot vectors, for categorical_crossentropy
Y_train = keras.utils.to_categorical(y_train, num_classes=nb_classes)
Y_test = keras.utils.to_categorical(y_test, num_classes=nb_classes)
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)
print(f"Label {y_train[0]} one-hot encoded: {Y_train[0]}")

## Multinomial logistic regression

Create with `keras.Sequential` a model without a hidden layer, `model`: a `keras.Input` of `input_dim` pixels, then a `layers.Dense` layer of `nb_classes` neurons, with the softmax activation. Each neuron computes a linear combination of the pixels, and the softmax turns them into 10 probabilities that sum to 1: this is a multinomial logistic regression.

Display its summary with `summary()` and explain its number of parameters.

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `keras.Sequential([...])` takes the list of layers, in order
- `keras.Input(shape=(input_dim,))`: the shape of **one** example, without the batch size
- `layers.Dense(nb_classes, activation="softmax")`
- Expected result: 7,850 parameters

A skeleton to copy into the exercise cell, replacing each `...`:

```python
model = keras.Sequential([keras.Input(shape=...), layers.Dense(..., activation=...)])
model.summary()
```

### Solution

In [ ]:
model = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(nb_classes, activation="softmax")]
)
model.summary()

# A Dense layer has one weight per input and per neuron, plus one bias per neuron:
# 784 × 10 + 10 = 7,850 parameters. None is the batch size, left free.

## Training

Compile the model with `compile`: the `"sgd"` optimizer (stochastic gradient descent, learning rate of 0.01), the `categorical_crossentropy` loss, which takes the one-hot targets `Y_train`, and the `"accuracy"` metric. Then train it with `fit`, with:

- 128 examples per batch;
- 10 epochs;
- 20% of the training data for validation (`validation_split`).

Finally, measure its loss and accuracy on the test set with the `evaluate` function below. Keep these numbers: they are the baseline for the next models.

In [ ]:
def evaluate(model: keras.Model, one_hot: bool = False) -> None:
  """Print the loss and accuracy of `model` on the test set.

  one_hot: True for a model trained on the one-hot targets (Y_train).
  """
  loss, accuracy = model.evaluate(X_test, Y_test if one_hot else y_test, verbose=0)
  print(f"Test: loss {loss:.3f}, accuracy {accuracy:.3f}")

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `model.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])`
- `model.fit(X_train, Y_train, batch_size=128, epochs=10, validation_split=0.2)` returns a `History` object, whose `history` attribute holds the loss and metrics of each epoch
- `evaluate(model, one_hot=True)`: the model learns on one-hot targets
- Expected result: about 89% accuracy in validation as in test

A skeleton to copy into the exercise cell, replacing each `...`:

```python
model.compile(optimizer=..., loss=..., metrics=[...])
history = model.fit(X_train, ..., batch_size=..., epochs=..., validation_split=...)
evaluate(model, one_hot=True)
```

### Solution

In [ ]:
# compile attaches an optimizer, a loss function and metrics to the model
model.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])

# fit trains the compiled model: a complete training loop
history = model.fit(X_train, Y_train, batch_size=128, epochs=10, validation_split=0.2)
evaluate(model, one_hot=True)

## A loss for integer labels

The `sparse_categorical_crossentropy` loss computes the same cross-entropy, but from the integer labels `y_train`: one-hot encoding becomes unnecessary.

Create a model `model2` identical to the previous one, compile it with this loss, and train it on `y_train` with the same settings. Compare its test loss and accuracy with those of the first model: what does the label format change?

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- the same `keras.Sequential` as `model`: a new model, with fresh weights
- `loss="sparse_categorical_crossentropy"`, then `fit` on `y_train`, and `evaluate(model2)`, without `one_hot`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
model2 = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(nb_classes, activation="softmax")]
)
model2.compile(optimizer="sgd", loss=..., metrics=["accuracy"])
model2.fit(X_train, ..., batch_size=128, epochs=10, validation_split=0.2)
evaluate(model2)
```

### Solution

In [ ]:
model2 = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(nb_classes, activation="softmax")]
)
model2.compile(
  optimizer="sgd", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
model2.fit(X_train, y_train, batch_size=128, epochs=10, validation_split=0.2)
evaluate(model2)

# The same results, up to random draws (initialization, batch order): only the
# format of the targets changes. Integer labels avoid storing 10 values per
# example.

## Regularized multilayer perceptron

Write a function `build_deep_model` that returns a fresh, uncompiled multilayer perceptron:

- 4 hidden layers of 20 neurons, with the ReLU activation;
- on their weights, an L2 penalty with a coefficient of $10^{-4}$ (`kernel_regularizer`) and an orthogonal initialization (`kernel_initializer`);
- the same softmax output layer as the logistic regression.

Create one, `deep_model`, display its summary, compile it with the `"adam"` optimizer, the `sparse_categorical_crossentropy` loss and the accuracy, then train it with the same settings as the logistic regression. How many points of test accuracy does it gain over it? For how many more parameters?

The next parts reuse `build_deep_model`.

**Useful documentation**

- [`keras.initializers.Orthogonal`](https://keras.io/api/layers/initializers/#orthogonal-class)

In [ ]:
def build_deep_model() -> keras.Model: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `kernel_regularizer=keras.regularizers.L2(1e-4)` adds $10^{-4} \sum w^2$ to the loss, over the layer's weights; the biases are not penalized
- `kernel_initializer="orthogonal"`: orthogonal initial weight matrices, which preserve the norm of the signal from one layer to the next
- The four hidden layers have the same options: a dictionary `hidden`, passed with `**hidden`, avoids repeating them
- Expected result: 17,170 parameters, about 95% test accuracy

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def build_deep_model() -> keras.Model:
  hidden = dict(activation=..., kernel_regularizer=..., kernel_initializer=...)
  return keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(20, **hidden),
      ...,
      layers.Dense(nb_classes, activation="softmax"),
    ]
  )


deep_model = build_deep_model()
deep_model.summary()
deep_model.compile(optimizer=..., loss=..., metrics=["accuracy"])
deep_model.fit(X_train, y_train, batch_size=128, epochs=10, validation_split=0.2)
evaluate(deep_model)
```

### Solution

In [ ]:
def build_deep_model() -> keras.Model:
  """A fresh multilayer perceptron: 4 regularized hidden layers of 20 neurons."""
  hidden = dict(
    activation="relu",
    kernel_regularizer=keras.regularizers.L2(1e-4),
    kernel_initializer="orthogonal",
  )
  return keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(20, **hidden),
      layers.Dense(20, **hidden),
      layers.Dense(20, **hidden),
      layers.Dense(20, **hidden),
      layers.Dense(nb_classes, activation="softmax"),
    ],
    name="deep_model",
  )


deep_model = build_deep_model()
deep_model.summary()
deep_model.compile(
  optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
deep_model.fit(X_train, y_train, batch_size=128, epochs=10, validation_split=0.2)
evaluate(deep_model)

# About 6 points more than the logistic regression (95% against 89%), for 17,170
# parameters instead of 7,850. Part of the gain comes from Adam: the TensorBoard
# part compares the two optimizers on this model.

## Hyperparameter search

How many neurons in the hidden layer, which learning rate? KerasTuner tries combinations of hyperparameters and keeps the best one, scored on the validation set.

Write a function `build_model(hp)` that returns a perceptron with one ReLU hidden layer, compiled with Adam, the `sparse_categorical_crossentropy` loss and the accuracy. It reads its hyperparameters from `hp`:

- `units`, the number of hidden neurons: from 32 to 512, in steps of 32;
- `learning_rate`, Adam's learning rate: from $10^{-4}$ to $10^{-2}$, on a logarithmic scale.

Then run the search with `keras_tuner.BayesianOptimization`: unlike `RandomSearch`, which draws each trial at random, it picks the next one from the results of the previous ones. Objective `val_accuracy`, 5 trials, each trained 3 times for a less noisy average, 10 epochs per training, batches of 1,500 examples and 20% validation. Finally, display the summary of the trials, then the best model, with its test loss and accuracy.

**Useful documentation**

- [`keras_tuner.HyperParameters`](https://keras.io/keras_tuner/api/hyperparameters/)
- [`keras_tuner.BayesianOptimization`](https://keras.io/keras_tuner/api/tuners/bayesian/#bayesianoptimization-class)

In [ ]:
def build_model(hp: keras_tuner.HyperParameters) -> keras.Model: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `hp.Int("units", min_value=32, max_value=512, step=32)` and `hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")` return the value of the current trial
- `keras_tuner.BayesianOptimization(build_model, objective="val_accuracy", max_trials=5, executions_per_trial=3, seed=0, directory="tuner", project_name="mnist", overwrite=True)`: `overwrite=True` starts over instead of resuming the search already saved in `tuner/mnist`
- `tuner.search(...)` takes the arguments of `fit`
- `tuner.results_summary()` displays the trials, from best to worst; `tuner.get_best_models()[0]` returns the best model, already trained

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def build_model(hp: keras_tuner.HyperParameters) -> keras.Model:
  units = hp.Int("units", ...)
  learning_rate = hp.Float("learning_rate", ...)
  model = keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(..., activation="relu"),
      layers.Dense(nb_classes, activation="softmax"),
    ]
  )
  model.compile(
    optimizer=keras.optimizers.Adam(...),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  return model


tuner = keras_tuner.BayesianOptimization(build_model, ...)
tuner.search(X_train, y_train, ...)
tuner.results_summary()
best_model = tuner.get_best_models()[0]
best_model.summary()
evaluate(best_model)
```

### Solution

In [ ]:
def build_model(hp: keras_tuner.HyperParameters) -> keras.Model:
  # Each call to hp reads the value the tuner picked for the current trial
  units = hp.Int("units", min_value=32, max_value=512, step=32)
  learning_rate = hp.Float(
    "learning_rate", min_value=1e-4, max_value=1e-2, sampling="log"
  )
  model = keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(units, activation="relu"),
      layers.Dense(nb_classes, activation="softmax"),
    ]
  )
  model.compile(
    optimizer=keras.optimizers.Adam(learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  return model


tuner = keras_tuner.BayesianOptimization(
  build_model,
  objective="val_accuracy",
  max_trials=5,
  executions_per_trial=3,
  seed=0,
  directory="tuner",
  project_name="mnist",
  overwrite=True,
)
tuner.search_space_summary()

In [ ]:
tuner.search(X_train, y_train, epochs=10, batch_size=1500, validation_split=0.2)

In [ ]:
tuner.results_summary()
best_model = tuner.get_best_models()[0]
best_model.summary()
evaluate(best_model)

## Adam against SGD in TensorBoard

TensorBoard displays the learning curves that a `keras.callbacks.TensorBoard` callback, passed to `fit`, writes into a directory: one subdirectory per training. Run the next cell before the exercise: the dashboard fills up as the epochs go (reload button at the top right).

Then train two fresh multilayer perceptrons (`build_deep_model()`) with the settings of the "Regularized multilayer perceptron" part, one with the `"adam"` optimizer, the other with `"sgd"`. Pass to each `fit` a TensorBoard callback that writes into `logs/adam` for the first one, into `logs/sgd` for the second one.

Compare the `epoch_loss` and `epoch_accuracy` curves of the two trainings: which one converges faster? Why?

**Useful documentation**

- [`keras.callbacks.TensorBoard`](https://keras.io/api/callbacks/tensorboard/)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `keras.callbacks.TensorBoard(log_dir="logs/adam")`, passed through `fit(..., callbacks=[...])`
- A fresh model per optimizer, `build_deep_model()` then `compile`: otherwise the second training starts from the weights of the first one
- In TensorBoard, the *Scalars* tab overlays the curves of the two directories; `train` and `validation` separate the training and validation curves
- Running a training again adds its curves to the same directory: delete `logs` or change the directory name to start from a clean dashboard

A skeleton to copy into the exercise cell, replacing each `...`:

```python
for optimizer in ["adam", "sgd"]:
  model = build_deep_model()
  model.compile(
    optimizer=..., loss="sparse_categorical_crossentropy", metrics=["accuracy"]
  )
  tensorboard = keras.callbacks.TensorBoard(log_dir=...)
  model.fit(X_train, y_train, ..., callbacks=[...])
```

### Solution

In [ ]:
for optimizer in ["adam", "sgd"]:
  model = build_deep_model()
  model.compile(
    optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"]
  )
  tensorboard = keras.callbacks.TensorBoard(log_dir=f"logs/{optimizer}")
  model.fit(
    X_train,
    y_train,
    batch_size=128,
    epochs=10,
    validation_split=0.2,
    callbacks=[tensorboard],
  )

# Adam converges much faster: it exceeds 90% validation accuracy from the first
# epoch on, SGD only after several. SGD takes a step of the same size (0.01) for
# every weight; Adam adapts each weight's step to the size of its recent gradients,
# and keeps a momentum from one step to the next (the average of the gradients).

## Early stopping and saving the best model

Rather than setting the number of epochs in advance, we stop the training when the validation loss no longer decreases, and keep the best model seen.

Train a fresh multilayer perceptron with Adam, for at most 50 epochs, with two callbacks:

- `keras.callbacks.EarlyStopping`: stop after 3 epochs without a decrease of `val_loss`, and go back to the weights of the best epoch;
- `keras.callbacks.ModelCheckpoint`: save the model into `best.keras`, each time `val_loss` reaches a new minimum.

Display the number of epochs run. Then reload the saved model with `keras.models.load_model`, and compare its evaluation on the test set with that of the model in memory: why are they identical?

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)`
- `keras.callbacks.ModelCheckpoint("best.keras", monitor="val_loss", save_best_only=True)`
- `fit(..., epochs=50, callbacks=[...])` returns a `History`: `len(history.history["loss"])` is the number of epochs run
- `keras.models.load_model("best.keras")` returns a compiled model, ready for `evaluate`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
model = build_deep_model()
model.compile(
  optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
callbacks = [
  keras.callbacks.EarlyStopping(...),
  keras.callbacks.ModelCheckpoint(...),
]
history = model.fit(X_train, y_train, ..., callbacks=callbacks)
print(len(...))
evaluate(model)
evaluate(keras.models.load_model(...))
```

### Solution

In [ ]:
model = build_deep_model()
model.compile(
  optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
callbacks = [
  # Stop after 3 epochs without a decrease of val_loss, best epoch's weights restored
  keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
  ),
  # Save the model at each new minimum of val_loss
  keras.callbacks.ModelCheckpoint(
    "best.keras", monitor="val_loss", save_best_only=True
  ),
]
history = model.fit(
  X_train,
  y_train,
  batch_size=128,
  epochs=50,
  validation_split=0.2,
  callbacks=callbacks,
)
print(f"Epochs run: {len(history.history['loss'])}")

evaluate(model)
best_model = keras.models.load_model("best.keras")
evaluate(best_model)

# The same numbers: restore_best_weights puts back into the model the weights of
# the best epoch, the one ModelCheckpoint saved last. Without it, the model in
# memory would keep the weights of the last epoch, 3 epochs after the best one.